# Structural, Chemical, and Computational Data on FOXM1 Forkhead Domain Inhibitors Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² dataset using the `mlcroissant` library. All entities in the dataset, including record sets, fields, and columns, are referenced by their unique `@id` values as per the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.5419-b4nt/fair2.json](https://sen.science/doi/10.71728/senscience.5419-b4nt/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.5419-b4nt/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Display dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list available record set `@id`s, then fields for each record set.

In [ ]:
# List all record sets and their @id values
record_sets = [record_set['@id'] for record_set in metadata.to_json().get('recordSet', [])]
print("Available Record Sets (@id):")
for rs_id in record_sets:
    print(f"- Record Set @id: {rs_id}")

# For each record set, print its fields and columns by @id
for rs_id in record_sets:
    print(f"\nRecord Set: {rs_id}")
    records = dataset.records(record_set=rs_id)
    try:
        first_record = next(records)
        print("Fields in first record (@id):")
        for field_id in first_record.keys():
            print(f"  - {field_id}")
    except StopIteration:
        print("No records found.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Record set and field/column `@id`s are used throughout.

In [ ]:
# Extract data from each record set into a DataFrame
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns for Record Set {record_set_id}: {df.columns.tolist()}")
        print(f"Sample Data for {record_set_id}:")
        display(df.head())
    else:
        print(f"No records available for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll pick a record set with data, select a numeric field (by its `@id`), filter, normalize, and group.

In [ ]:
# Choose a record set with data for analysis
if dataframes:
    chosen_record_set_id = list(dataframes.keys())[0]
    chosen_df = dataframes[chosen_record_set_id]

    # Identify numeric fields
    numeric_fields = [col for col in chosen_df.columns if pd.api.types.is_numeric_dtype(chosen_df[col])]
    print(f"Numeric Fields (@id) in {chosen_record_set_id}: {numeric_fields}")

    # Use the first numeric field
    if numeric_fields:
        numeric_field_id = numeric_fields[0]

        # EDA: Filter, normalize, and group
        threshold = chosen_df[numeric_field_id].mean()
        filtered_df = chosen_df[chosen_df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Identify a potential group field
        group_fields = [col for col in chosen_df.columns if chosen_df[col].dtype == 'object' and col != numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No DataFrames available; cannot perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field, and a scatter plot if a suitable categorical grouping exists.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_fields:
    # Histogram of numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(chosen_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {chosen_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field exists, boxplot
    if group_fields:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=chosen_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset provides rich structural, chemical, and computational data on FOXM1 Forkhead Domain Inhibitors.
- Using the `mlcroissant` library, we loaded metadata and records, explored available record sets and fields via their `@id`s.
- Data extraction and EDA demonstrated filtering, normalization, and grouping capabilities using unique identifiers.
- Visualizations reveal distributions and relationships in the dataset, supporting further drug discovery and computational modeling research.

Further analysis may include advanced modeling or integration with external resources, leveraging the Croissant schema structure for reproducible FAIR data workflows.